# Занятие 3, пара 1. Парсинг. requests и BeautifulSoup


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


Сегодня у нас будет маленький переезд: данные живут на странице, а нам хочется поселить их в таблице. Давайте разберёмся, как это сделать без ручного копирования каждой цены.

Понадобятся Python, pandas и, по основному маршруту, BeautifulSoup. Интернет и Git не нужны. После пары вы сможете найти карточки в HTML, извлечь их поля и проверить, что таблица соответствует странице.


### Подготовка

Работайте в своей копии внутри папки проекта. Запустите ячейки по порядку. Эта тетрадь не использует переменные из другой тетради. GitHub — по желанию; сохраняйте файл и отдельную копию.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check

PROJECT_DIR


In [ ]:
import pandas as pd
from IPython.display import HTML, display
from course_support.parsing import parse_catalog_stdlib
try:
    from bs4 import BeautifulSoup
except ImportError:
    BeautifulSoup = None
    print('BeautifulSoup недоступна: используем резерв для учебного каталога.')
CATALOG_DIR = DATA_DIR / 'parsing'
OUTPUT_DIR = WORKSPACE_DIR / 'reports' / 'parsing'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
html = (CATALOG_DIR / 'page-1.html').read_text(encoding='utf-8')
print('Символов в HTML:', len(html))


## Сначала посмотрим глазами
Зачастую данные можно найти в готовом виде. Но иногда они есть только на странице, которую удобно читать человеку, а не анализировать программой. Тут пригодится парсинг — извлечение структуры из текста.

Перед вами **учебный каталог**, не реальный рынок недвижимости. Все объявления и адреса вымышлены. Одно объявление станет одной строкой таблицы. Какой столбец поможет отличить повтор объявления от новой квартиры?


In [ ]:
# Ссылки во встроенном показе отключены; настоящий каталог можно открыть браузером.
display(HTML(html.replace('href=', 'data-href=')))


### HTML — это дерево, а не суп из символов
У каждого объявления свой `<article class="listing" data-id="…">`. Внутри — ссылка с названием и абзацы с адресом, ценой и площадью. Текст между тегами и атрибуты тегов — разные вещи.

Откройте page-1.html в браузере и выберите «Исследовать элемент» / «Просмотреть код элемента». Названия меню могут отличаться. Если неудобно, следующая ячейка покажет тот же фрагмент. Ищем одну карточку, а не пытаемся прочитать весь HTML сразу.


In [ ]:
start = html.index('<article')
print(html[start:html.index('</article>', start) + len('</article>')])


## Найдём первую карточку
План простой: прочитать HTML → найти карточки → собрать словари → получить DataFrame.

`find` возвращает первый подходящий тег, `find_all` — все подходящие. `select` и `select_one` принимают CSS-селекторы: `.listing` означает класс, `article.listing` — тег article с этим классом. Парсер `html.parser` входит в Python; отдельный lxml сегодня не нужен.

Если библиотека не найдёт тег, `find`/`select_one` вернут `None`. Это повод проверить разметку, а не добавлять наугад ещё один индекс.


In [ ]:
if BeautifulSoup is not None:
    soup = BeautifulSoup(html, 'html.parser')
    first = soup.find('article', class_='listing')
    print(first['data-id'])
    print(first.select_one('[data-field="title"]').get_text(' ', strip=True))
else:
    fallback_rows, _ = parse_catalog_stdlib(html)
    first = fallback_rows[0]
    print(first['listing_id'])
    print(first['title'])


### ✏️ Ваш ход 1: одна карточка

Сохраните ID первой карточки в `first_id`, название — в `first_title`. ID берём из атрибута, текст названия — из ссылки внутри карточки. Не перепечатывайте ответ вручную.
В резервном режиме достаньте поля `listing_id` и `title` из словаря `first`.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('03.1.1', first_id, first_title)


### Можно остановиться здесь
Сохраните тетрадь. Уже получилось: исходный HTML → нужная карточка → два поля. Продолжение начнём с короткого повторения этого шага.


## А теперь все карточки
Если собрать вообще все ссылки, в результат попадут меню и «Следующая». Нам нужны названия **внутри карточек**. Удобный короткий путь — CSS-селектор с пробелом: найти элементы внутри других элементов.


In [ ]:
if BeautifulSoup is not None:
    cards = soup.select('article.listing')
    print('Всех ссылок:', len(soup.find_all('a')))
    print('Карточек:', len(cards))
else:
    cards = fallback_rows
    print('Карточек:', len(cards))


### ✏️ Ваш ход 2: заголовки без меню

Получите список `titles` из названий всех четырёх карточек в порядке страницы. Можно циклом или списковым включением. Затем измените селектор в отдельной пробе на просто `a`: что лишнее вы соберёте? В резерве сравните список с HTML глазами.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('03.1.2', titles)


## Одна карточка — один словарь
Собирать отдельный список цен и отдельный список названий соблазнительно. Но если в одной карточке нет цены, списки могут съехать. Поэтому сначала берём карточку целиком, а уже внутри неё ищем поля.

Здесь функция только извлекает текст. «По запросу» не превращаем в ноль; числовые типы и пробелы разберём во второй паре. `href` сохраняем как указатель на источник записи.


In [ ]:
def record_from_card(card):
    if BeautifulSoup is None:
        return dict(card)
    return {
        'listing_id': card['data-id'],
        'title': card.select_one('[data-field="title"]').get_text(' ', strip=True),
        'address': card.select_one('[data-field="address"]').get_text(' ', strip=True),
        'price': card.select_one('[data-field="price"]').get_text(' ', strip=True),
        'area': card.select_one('[data-field="area"]').get_text(' ', strip=True),
        'href': card.select_one('[data-field="title"]')['href'],
    }
record_from_card(cards[0])


### ✏️ Ваш ход 3: наша первая выгрузка

Примените `record_from_card` ко всем карточкам и соберите `page_df` через `pd.DataFrame`. Покажите ID, название и цену. Убедитесь, что в таблице четыре строки, а ID не повторяются. Почему пока нельзя вычислить среднюю цену?


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('03.1.3', page_df)


In [ ]:
page_df.to_csv(OUTPUT_DIR / 'page1_raw.csv', index=False, encoding='utf-8')
print('Сохранено:', OUTPUT_DIR / 'page1_raw.csv')


## А если страница в интернете?
До сих пор мы читали файл. У HTTP-запроса добавляется транспорт: сервер может ответить ошибкой или не ответить вовремя. `requests.get` возвращает объект Response, а не строку; `.text` — текст ответа, `.status_code` — код, `.raise_for_status()` проверяет ошибки HTTP. Даже 200 ещё не доказывает, что пришёл нужный каталог: нужно проверить содержимое.

Следующая ячейка **по умолчанию не обращается в сеть**. Это необязательная проба на сайте, специально созданном для тренировки парсинга. Его HTML отличается от нашего: старые селекторы туда автоматически не переносятся. Основная работа уже сделана локально.


In [ ]:
RUN_ONLINE = False
if RUN_ONLINE:
    try:
        import requests
        response = requests.get('https://books.toscrape.com/', timeout=(3, 5),
                                headers={'User-Agent': 'UniversityPractice/1.0'})
        response.raise_for_status()
        print(response.status_code, response.headers.get('Content-Type'))
        print(response.text[:200])
    except ImportError:
        print('requests не установлена. Остаёмся на локальном примере.')
    except requests.RequestException as error:
        print('Запрос не удался; локальные данные доступны:', type(error).__name__)
else:
    print('Работаем с сохранённым HTML, запросов в интернет нет.')


## Практический запас — 15–20 минут
1. Поменяйте порядок атрибутов или добавьте пробелы в **копии строки HTML**. Селектор продолжает работать? Почему поиск по номеру строки был бы хуже?
2. Переименуйте `data-field="price"` в `data-field="cost"` в копии HTML. Найдите, какой селектор надо исправить. Не меняйте исходные файлы выдачи.
3. Вдвоём сформулируйте один вопрос к будущей таблице и необходимые поля. Сосед объясняет, каких полей не хватает. Тема проекта ещё не выбрана — используйте каталог.

## Перед закрытием
Сохраните тетрадь и CSV. Допишите два предложения: что является строкой и как проверили полноту первой страницы. Это тренировка, отдельной новой лабораторной и баллов нет. Дальше соберём остальные страницы и сохраним происхождение данных.

Справка: [Beautiful Soup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/), [Requests](https://requests.readthedocs.io/en/latest/user/quickstart/). Читать всю документацию к следующей паре не требуется.
